In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qubit-Initialisierung

<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Wenn ein Schaltkreis auf einer IBM&reg; Quantenprozessoreinheit (QPU) ausgeführt wird, wird am Anfang des Schaltkreises in der Regel ein impliziter Reset eingefügt, um sicherzustellen, dass die Qubits auf null initialisiert werden. Dies wird durch das Flag `init_qubits` gesteuert, das als [primitive Ausführungsoption](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2) gesetzt wird.

Der Reset-Prozess ist jedoch nicht perfekt und führt zu Zustandsvorbereitungsfehlern. Um diesen Fehler abzumildern, fügt das System außerdem eine Wiederholungsverzögerungszeit (oder `rep_delay`) zwischen den Schaltkreisen ein. Jedes Backend hat eine andere Standard-`rep_delay`, die in der Regel länger als T1 ist, damit die Umgebung die Qubits zurücksetzen kann. Die Standard-`rep_delay` kann durch Ausführen von `backend.default_rep_delay` abgefragt werden.

Alle IBM QPUs verwenden die dynamische Wiederholungsratenausführung, mit der du die `rep_delay` für jeden Job ändern kannst. Schaltkreise, die du in einem Primitive-Job einreichst, werden für die Ausführung auf der QPU zusammengefasst. Diese Schaltkreise werden ausgeführt, indem über die Schaltkreise für jeden angeforderten Shot iteriert wird; die Ausführung erfolgt spaltenweise über eine Matrix aus Schaltkreisen und Shots, wie in der folgenden Abbildung dargestellt.

![Die erste Spalte repräsentiert Shot 0. Die Schaltkreise werden der Reihe nach von 0 bis 3 ausgeführt. Die zweite Spalte repräsentiert Shot 1. Die Schaltkreise werden der Reihe nach von 0 bis 3 ausgeführt. Die verbleibenden Spalten folgen demselben Muster. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Spaltenweise Ausführungsmatrix")

Da `rep_delay` zwischen den Schaltkreisen eingefügt wird, tritt diese Verzögerung bei jedem Shot der Ausführung auf. Daher verringert ein niedrigerer `rep_delay`-Wert die gesamte QPU-Ausführungszeit, jedoch auf Kosten einer erhöhten Zustandsvorbereitungsfehlerrate, wie im folgenden Bild zu sehen ist:

![Dieses Bild zeigt, dass die Zustandsvorbereitungsfehlerrate mit sinkendem `rep_delay`-Wert zunimmt.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Wiederholungsverzögerung versus Fehlerrate")

Wenn du sowohl `rep_delay=0` als auch `init_qubits=False` setzt, werden die Schaltkreise im Wesentlichen „zusammengeführt", da die Qubits im Endzustand des vorherigen Shots beginnen.

Beachte, dass Schaltkreise in einem Primitive-Job zwar für die QPU-Ausführung zusammengefasst werden, die Reihenfolge der Ausführung der Schaltkreise aus PUBs jedoch nicht garantiert ist. Auch wenn du `pubs=[pub1, pub2]` einreichst, ist nicht garantiert, dass die Schaltkreise aus `pub1` vor denen aus `pub2` ausgeführt werden. Es ist ebenfalls nicht garantiert, dass Schaltkreise desselben Jobs als einzelner Batch auf der QPU ausgeführt werden.

## rep_delay für einen Primitive-Job festlegen

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Nächste Schritte
> **Tip:** - Probiere ein Beispiel im Tutorial zum [Quanten-Näherungsoptimierungsalgorithmus (QAOA)](/tutorials/quantum-approximate-optimization-algorithm) aus.
>   - Lies nach, wie du [mit Primitives loslegen kannst.](./get-started-with-primitives)